# 6 Ablation Single File

Sanity-check the six AMSTE ablation variants on a single file.

Paths are imported from `config.py` at the repository root.
Run the notebooks in numbered order; see the repository README.

In [ ]:
# --- repository configuration -------------------------------------
# All paths come from config.py at the repository root. Edit that
# file once; do not hardcode paths in this notebook.
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
import config
# ------------------------------------------------------------------

"""
DAS Microseismic SNN Encoding — AMSTE Ablation Study (Single File Diagnostic)
==============================================================================
This notebook tests AMSTE on a single SEGY file with leave-one-out ablation
of its four core components. It is used for visual diagnostics; downstream
classification metrics (F1, Recall, FNR, AUPRC) come from the SNN training
notebook (notebook 4).

  AMSTE = Adaptive Multi-Scale Spatio-Temporal Delta Encoder.
  It combines four DAS-specific properties:
      (1) Adaptive MAD threshold       — per-channel noise calibration
      (2) Polarity detection           — compression vs rarefaction
      (3) Multi-scale temporal voting  — coincidence across timescales
      (4) Spatial coherence gate       — single-channel noise suppression

  Ablation variants tested in this notebook (leave-one-out):
      [V1] Basic change encoder        — none of the 4 components
      [V2] AMSTE without MAD           — global fixed threshold (no per-channel)
      [V3] AMSTE without polarity      — abs(Δx) only (no +/− split)
      [V4] AMSTE without multi-scale   — single lag (lag = 1 only)
      [V5] AMSTE without spatial gate  — no spatial coherence check
      [V6] Full AMSTE                  — all four components active

  The full AMSTE configuration is expected to give the best balance between
  detection sensitivity (low FNR), spike efficiency (low energy / low spike
  rate), and event-window concentration (high precision).
"""

import segyio
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.signal  import butter, filtfilt, hilbert
from scipy.ndimage import maximum_filter1d, minimum_filter1d
from scipy.stats   import pearsonr
import os

# =============================================================================
# CONFIGURATION
# =============================================================================
FILE_PATH  = config.SINGLE_FILE_EVENT
OUTPUT_DIR = config.SINGLE_OUTPUT_DIR
os.makedirs(OUTPUT_DIR, exist_ok=True)

RNG_SEED = 42

# ── Preprocessing ─────────────────────────────────────────────────────────────
FORCE_DT_MS         = 0.5    # FRISCO FORGE 2024, 2 kHz DAS interrogator
FILTER_LOW_HZ       = 50
FILTER_HIGH_HZ      = 250
EXTRACT_WINDOW      = True
WINDOW_START_MS     = 0
WINDOW_END_MS       = 1000
TRACE_SPACING_M     = 4.0
ENERGY_PER_SPIKE_PJ = 23.6
EVENT_START_MS      = 380.0
EVENT_END_MS        = 560.0
# NOTE: Single-file diagnostic window only.
# Batch experiments use per-file labels from CSV — not this hard-coded window.
DISPLAY_TRACE       = 50

# ── AMSTE — Full method parameters (sweep-confirmed best) ────────────────────
AMSTE_ALPHA         = 3.0        # MAD multiplier for per-channel threshold
AMSTE_LAGS          = [1, 3, 8]  # temporal scales: 0.5, 1.5, 4.0 ms
AMSTE_MIN_VOTES     = 2          # majority vote (≥2/3 lags must agree)
AMSTE_WS            = 16         # spatial window: 16 channels (64 m aperture)
AMSTE_THR_S         = 0.5        # spatial amplitude gate threshold

# ── Variant colour map (V1..V6) ──────────────────────────────────────────────
ENCODER_COLORS = {
    'Basic':         '#9E9E9E',  # gray         — simplest baseline
    'No-MAD':        '#F44336',  # red          — fixed threshold
    'No-Polarity':   '#FF9800',  # orange       — |Δx| only
    'No-MultiScale': '#2196F3',  # blue         — single lag
    'No-Spatial':    '#4CAF50',  # green        — no spatial gate
    'Full AMSTE':    '#FF5722',  # deep orange  — full proposed method
}

# Component flags used to print the ablation matrix
# (MAD, Polarity, Multi-scale, Spatial-gate)
ABLATION_FLAGS = {
    'Basic':         (False, False, False, False),
    'No-MAD':        (False, True,  True,  True ),
    'No-Polarity':   (True,  False, True,  True ),
    'No-MultiScale': (True,  True,  False, True ),
    'No-Spatial':    (True,  True,  True,  False),
    'Full AMSTE':    (True,  True,  True,  True ),
}


# =============================================================================
# 1. DATA LOADING
# =============================================================================
def load_segy_file(filename, force_dt_ms=FORCE_DT_MS):
    try:
        with segyio.open(filename, mode='r', ignore_geometry=True) as f:
            data = np.array([f.trace[i] for i in range(len(f.trace))])
            try:
                dt_ms = f.bin[segyio.BinField.Interval] / 1000.0
                if not (0.001 <= dt_ms <= 10.0):
                    print(f"  Header DT invalid → forced: {force_dt_ms} ms")
                    dt_ms = force_dt_ms
                else:
                    print(f" Header DT: {dt_ms} ms")
            except Exception:
                print(f"  Cannot read DT → forced: {force_dt_ms} ms")
                dt_ms = force_dt_ms
            print(f" {data.shape[0]} traces × {data.shape[1]} samples "
                  f"| {dt_ms} ms ({1000/dt_ms:.0f} Hz)")
            return data, dt_ms
    except Exception as e:
        print(f" {e}"); return None, None


# =============================================================================
# 2. PREPROCESSING
# =============================================================================
def bandpass_filter(data, dt_ms, lo=FILTER_LOW_HZ, hi=FILTER_HIGH_HZ):
    fs  = 1000.0 / dt_ms
    nyq = 0.5 * fs
    b, a = butter(4, [np.clip(lo/nyq, 1e-6, 0.99),
                      np.clip(hi/nyq, 1e-6, 0.99)], btype='band')
    if data.ndim == 1:
        return filtfilt(b, a, data)
    out = np.zeros_like(data)
    for i in range(data.shape[0]):
        out[i] = filtfilt(b, a, data[i])
    return out

def extract_time_window(data, dt_ms, s_ms, e_ms):
    s = max(0, int(s_ms / dt_ms))
    e = min(data.shape[1], int(e_ms / dt_ms))
    return (data, 0, data.shape[1]) if s >= e else (data[:, s:e], s, e)

def normalize_unit(data):
    """Unit normalisation → [0, 1] per trace. Used for envelope reference."""
    out = np.zeros_like(data)
    for i in range(data.shape[0]):
        t = data[i] - np.mean(data[i])
        m = np.max(np.abs(t))
        out[i] = (t / m + 1) / 2 if m > 0 else 0.5
    return out

def normalize_signed(data):
    """
    Signed normalisation → output range [-1, +1] per trace.
    AMSTE requires signed input so that bidirectional polarity voting works:
    a −0.8 strain excursion must map to −0.8 (visible to neg-vote), not to
    0.1 as it would under unit [0,1] normalisation.
    """
    out = np.zeros_like(data, dtype=np.float64)
    for i in range(data.shape[0]):
        tr = data[i] - np.mean(data[i])
        m  = np.max(np.abs(tr))
        out[i] = tr / m if m > 0 else tr
    return out

def extract_envelope(data):
    env = np.zeros_like(data)
    for i in range(data.shape[0]):
        e = np.abs(hilbert(data[i]))
        m = np.max(e)
        env[i] = e / m if m > 0 else e
    return env


# =============================================================================
# 3. AMSTE ABLATION VARIANTS  (V1 … V6)
# =============================================================================
# All variants take signed-normalised input in [-1, +1].
# All variants return a binary {0, 1} spike map of the same shape.
# Variants share parameter values where applicable, so any difference in
# output is attributable to the ablated component, not to retuning.
# -----------------------------------------------------------------------------

def _per_channel_mad_threshold(data, alpha):
    """Per-channel MAD threshold:  θ_c = α · 1.4826 · MAD_c    (n_tr, 1)."""
    med_c = np.median(data, axis=1, keepdims=True)
    mad_c = np.median(np.abs(data - med_c), axis=1, keepdims=True)
    return np.maximum(alpha * 1.4826 * mad_c, 1e-9)

def _global_mad_threshold(data, alpha):
    """Global MAD threshold: one scalar shared across all channels."""
    med = np.median(data)
    mad = np.median(np.abs(data - med))
    return float(max(alpha * 1.4826 * mad, 1e-9))

def _spatial_da(data, ws):
    """Spatial amplitude-spread:  max(|x|) − min(|x|) over a ws-channel window."""
    abs_data = np.abs(data)
    spat_max = maximum_filter1d(abs_data, size=ws, axis=0, mode='nearest')
    spat_min = minimum_filter1d(abs_data, size=ws, axis=0, mode='nearest')
    return spat_max - spat_min


# ── [V1] Basic change encoder ────────────────────────────────────────────────
def basic_change_encoding(data, dt_ms, alpha=AMSTE_ALPHA):
    """
    [V1] Basic change encoder — none of the 4 AMSTE components.
        spike[c, t] = 1  if  |x[c, t] − x[c, t−1]| > θ_global
    Fixed global threshold, single lag, no polarity, no spatial gate.
    Simplest possible delta encoder; serves as the lower-bound baseline.
    """
    theta = _global_mad_threshold(data, alpha)
    diff  = np.abs(np.diff(data, axis=1, prepend=data[:, :1]))
    out   = (diff > theta).astype(np.float32)
    sp    = float(np.sum(out) / out.size)
    print(f"   [V1] Basic change       sparsity: {sp:.2%}  "
          f"(fixed θ={theta:.4f}, lag=1, |Δx|, no spatial)")
    return out


# ── [V2] AMSTE without MAD ───────────────────────────────────────────────────
def amste_no_mad(data, dt_ms,
                 alpha=AMSTE_ALPHA, lags=None,
                 min_votes=AMSTE_MIN_VOTES, ws=AMSTE_WS, thr_s=AMSTE_THR_S):
    """
    [V2] AMSTE without adaptive MAD threshold.
    Single GLOBAL threshold replaces the per-channel θ_c.
    Polarity, multi-scale voting, and spatial gate are kept.
    Tests whether channel-wise noise calibration is needed.
    """
    if lags is None: lags = AMSTE_LAGS
    n_tr, n_smp = data.shape
    theta = _global_mad_threshold(data, alpha)   # scalar — same θ for all channels

    pos_votes = np.zeros((n_tr, n_smp), dtype=np.int32)
    neg_votes = np.zeros((n_tr, n_smp), dtype=np.int32)
    for lag in lags:
        diff_l = data[:, lag:] - data[:, :-lag]
        pos_votes[:, lag:] += (diff_l >  theta).astype(np.int32)
        neg_votes[:, lag:] += (diff_l < -theta).astype(np.int32)
    candidate = (pos_votes >= min_votes) | (neg_votes >= min_votes)

    da_s = _spatial_da(data, ws)
    out  = (candidate & (da_s > thr_s)).astype(np.float32)
    sp   = float(np.sum(out) / out.size)
    print(f"   [V2] No-MAD             sparsity: {sp:.2%}  "
          f"(fixed θ={theta:.4f}, +polarity, +multi, +spatial)")
    return out


# ── [V3] AMSTE without polarity ──────────────────────────────────────────────
def amste_no_polarity(data, dt_ms,
                      alpha=AMSTE_ALPHA, lags=None,
                      min_votes=AMSTE_MIN_VOTES, ws=AMSTE_WS, thr_s=AMSTE_THR_S):
    """
    [V3] AMSTE without polarity separation.
    Replaces the (+vote, −vote) pair with a single |Δx| > θ_c condition.
    MAD threshold, multi-scale voting, and spatial gate are kept.
    Tests whether compression-vs-rarefaction direction information matters.
    """
    if lags is None: lags = AMSTE_LAGS
    n_tr, n_smp = data.shape
    theta_c = _per_channel_mad_threshold(data, alpha)

    votes = np.zeros((n_tr, n_smp), dtype=np.int32)
    for lag in lags:
        diff_l = np.abs(data[:, lag:] - data[:, :-lag])
        votes[:, lag:] += (diff_l > theta_c).astype(np.int32)
    candidate = (votes >= min_votes)

    da_s = _spatial_da(data, ws)
    out  = (candidate & (da_s > thr_s)).astype(np.float32)
    sp   = float(np.sum(out) / out.size)
    print(f"   [V3] No-Polarity        sparsity: {sp:.2%}  "
          f"(+MAD, |Δx|, +multi, +spatial)")
    return out


# ── [V4] AMSTE without multi-scale ───────────────────────────────────────────
def amste_no_multiscale(data, dt_ms,
                         alpha=AMSTE_ALPHA,
                         ws=AMSTE_WS, thr_s=AMSTE_THR_S):
    """
    [V4] AMSTE without multi-scale temporal voting.
    Uses only lag = 1 (single timescale, no majority vote).
    MAD threshold, polarity, and spatial gate are kept.
    Tests whether multi-scale coincidence reduces missed events.
    """
    n_tr, n_smp = data.shape
    theta_c = _per_channel_mad_threshold(data, alpha)

    diff_l = data[:, 1:] - data[:, :-1]
    pos = np.zeros((n_tr, n_smp), dtype=bool)
    neg = np.zeros((n_tr, n_smp), dtype=bool)
    pos[:, 1:] = (diff_l >  theta_c)
    neg[:, 1:] = (diff_l < -theta_c)
    candidate = (pos | neg)

    da_s = _spatial_da(data, ws)
    out  = (candidate & (da_s > thr_s)).astype(np.float32)
    sp   = float(np.sum(out) / out.size)
    print(f"   [V4] No-MultiScale      sparsity: {sp:.2%}  "
          f"(+MAD, +polarity, lag=1 only, +spatial)")
    return out


# ── [V5] AMSTE without spatial gate ──────────────────────────────────────────
def amste_no_spatial(data, dt_ms,
                     alpha=AMSTE_ALPHA, lags=None,
                     min_votes=AMSTE_MIN_VOTES):
    """
    [V5] AMSTE without spatial coherence gate.
    Skips the neighbourhood amplitude-spread check.
    MAD threshold, polarity, and multi-scale voting are kept.
    Tests whether spatial filtering suppresses isolated DAS noise.
    """
    if lags is None: lags = AMSTE_LAGS
    n_tr, n_smp = data.shape
    theta_c = _per_channel_mad_threshold(data, alpha)

    pos_votes = np.zeros((n_tr, n_smp), dtype=np.int32)
    neg_votes = np.zeros((n_tr, n_smp), dtype=np.int32)
    for lag in lags:
        diff_l = data[:, lag:] - data[:, :-lag]
        pos_votes[:, lag:] += (diff_l >  theta_c).astype(np.int32)
        neg_votes[:, lag:] += (diff_l < -theta_c).astype(np.int32)
    candidate = (pos_votes >= min_votes) | (neg_votes >= min_votes)

    out = candidate.astype(np.float32)
    sp  = float(np.sum(out) / out.size)
    print(f"   [V5] No-Spatial         sparsity: {sp:.2%}  "
          f"(+MAD, +polarity, +multi, no spatial)")
    return out


# ── [V6] Full AMSTE ──────────────────────────────────────────────────────────
def amste_full(data, dt_ms,
               alpha=AMSTE_ALPHA, lags=None,
               min_votes=AMSTE_MIN_VOTES, ws=AMSTE_WS, thr_s=AMSTE_THR_S):
    """
    [V6] Full AMSTE — all four components active.

    STEP 1 — Per-channel MAD threshold
        MAD_c = median(|X[c,:] − median(X[c,:])|)
        θ_c   = α · 1.4826 · MAD_c
    STEP 2 — Bidirectional polarity at multiple lags
        ΔX_L[c,t] = X[c,t] − X[c,t−L]    for L in lags
        +vote when ΔX_L >  +θ_c          (compression)
        −vote when ΔX_L <  −θ_c          (rarefaction)
    STEP 3 — Multi-scale majority vote (per polarity)
        candidate(c,t) = (pos_votes ≥ min_votes) ∨ (neg_votes ≥ min_votes)
    STEP 4 — Spatial coherence gate
        da_s[c,t] = max(|X[c−ws/2:c+ws/2, t]|) − min(...)
        keep candidate only if da_s > θ_s

    OUTPUT: binary {0, 1}, compatible with all SNN training notebooks.
    """
    if lags is None: lags = AMSTE_LAGS
    n_tr, n_smp = data.shape
    theta_c = _per_channel_mad_threshold(data, alpha)

    pos_votes = np.zeros((n_tr, n_smp), dtype=np.int32)
    neg_votes = np.zeros((n_tr, n_smp), dtype=np.int32)
    for lag in lags:
        diff_l = data[:, lag:] - data[:, :-lag]
        pos_votes[:, lag:] += (diff_l >  theta_c).astype(np.int32)
        neg_votes[:, lag:] += (diff_l < -theta_c).astype(np.int32)
    candidate = (pos_votes >= min_votes) | (neg_votes >= min_votes)

    da_s = _spatial_da(data, ws)
    out  = (candidate & (da_s > thr_s)).astype(np.float32)
    sp   = float(np.sum(out) / out.size)
    print(f"   [V6] Full AMSTE         sparsity: {sp:.2%}  "
          f"(+MAD, +polarity, +multi, +spatial)")
    return out


# =============================================================================
# 4. BUILD ALL ABLATION VARIANTS
# =============================================================================
def _build_amste_ablation_variants(norm_signed, dt_ms):
    """
    Build the 6 ablation variants. All variants receive identical input
    (norm_signed, dt_ms) and use the same hyper-parameters where applicable.
    Any output difference is attributable to the ablated component alone.
    """
    print("\n    Building 6 AMSTE ablation variants …")
    print("   ── Leave-one-out ablation ─────────────────────────────────")
    variants = {
        'Basic':         basic_change_encoding(norm_signed, dt_ms),
        'No-MAD':        amste_no_mad(norm_signed,        dt_ms),
        'No-Polarity':   amste_no_polarity(norm_signed,   dt_ms),
        'No-MultiScale': amste_no_multiscale(norm_signed, dt_ms),
        'No-Spatial':    amste_no_spatial(norm_signed,    dt_ms),
        'Full AMSTE':    amste_full(norm_signed,          dt_ms),
    }
    return variants


# =============================================================================
# 5. METRICS
# =============================================================================
def spike_density_profile(spikes, dt_ms, smooth_ms=20.0):
    stacked     = np.mean(np.abs(spikes), axis=0)
    kernel_size = max(3, int(smooth_ms / dt_ms))
    kernel      = np.ones(kernel_size) / kernel_size
    return np.convolve(stacked, kernel, mode='same')

def temporal_fidelity(spikes, envelope, dt_ms, smooth_ms=20.0):
    spike_profile    = spike_density_profile(spikes, dt_ms, smooth_ms)
    envelope_stacked = np.mean(envelope, axis=0)
    kernel_size      = max(3, int(smooth_ms / dt_ms))
    kernel           = np.ones(kernel_size) / kernel_size
    env_smooth       = np.convolve(envelope_stacked, kernel, mode='same')
    r, _             = pearsonr(spike_profile, env_smooth)
    return float(r)

def event_snr(spikes, dt_ms,
              event_start_ms=EVENT_START_MS, event_end_ms=EVENT_END_MS):
    n_tr, n = np.abs(spikes).shape
    s       = max(0, int(event_start_ms / dt_ms))
    e       = min(n, int(event_end_ms   / dt_ms))
    density     = np.mean(np.abs(spikes), axis=0)
    in_event    = np.mean(density[s:e])
    out_samples = np.concatenate([density[:s], density[e:]])
    min_density = 1.0 / (n_tr * n)
    out_event   = max(np.mean(out_samples), min_density) if len(out_samples) > 0 else min_density
    return float(in_event / out_event)

def event_precision(spikes, dt_ms,
                    event_start_ms=EVENT_START_MS, event_end_ms=EVENT_END_MS):
    n = spikes.shape[1]
    s = max(0, int(event_start_ms / dt_ms))
    e = min(n, int(event_end_ms   / dt_ms))
    total = np.sum(np.abs(spikes))
    return float(np.sum(np.abs(spikes[:, s:e])) / total) if total > 0 else 0.0

def onset_detection_latency(spikes, dt_ms,
                             event_start_ms=EVENT_START_MS, search_ms=100.0):
    s = max(0, int(event_start_ms / dt_ms))
    e = min(spikes.shape[1], int((event_start_ms + search_ms) / dt_ms))
    latencies = []
    for t in range(spikes.shape[0]):
        idx = np.where(np.abs(spikes[t, s:e]) > 0)[0]
        if len(idx):
            latencies.append(idx[0] * dt_ms)
    return float(np.mean(latencies)) if latencies else None

def energy_efficiency(spikes, fidelity):
    energy_uj = float(np.sum(np.abs(spikes)) * ENERGY_PER_SPIKE_PJ / 1e6)
    return float(fidelity / (energy_uj + 1e-9))

def compute_all_metrics(spikes, envelope, dt_ms, name):
    fidelity  = temporal_fidelity(spikes, envelope, dt_ms)
    snr       = event_snr(spikes, dt_ms)
    precision = event_precision(spikes, dt_ms)
    latency   = onset_detection_latency(spikes, dt_ms)
    sp        = float(np.sum(np.abs(spikes)) / spikes.size)
    energy_uj = float(np.sum(np.abs(spikes)) * ENERGY_PER_SPIKE_PJ / 1e6)
    eff       = energy_efficiency(spikes, fidelity)
    print(f"\n  [{name:13}] Fidelity={fidelity:.3f}  SNR={snr:.2f}×  "
          f"Precision={precision:.3f}  Sparsity={sp:.2%}  Energy={energy_uj:.3f}µJ")
    return {'fidelity': fidelity, 'snr': snr, 'precision': precision,
            'latency_ms': latency, 'sparsity': sp, 'energy_uj': energy_uj,
            'efficiency': eff}


# =============================================================================
# 6. ABLATION EXPERIMENT
# =============================================================================
def run_ablation_experiment(all_variants, norm_envelope, dt_ms):
    print(f"\n{'='*78}")
    print("AMSTE ABLATION — EVENT CHARACTERISATION METRICS (single-file diagnostic)")
    print(f"{'='*78}")
    results = {}
    for name, spk in all_variants.items():
        results[name] = compute_all_metrics(spk, norm_envelope, dt_ms, name)

    # ── Component matrix + metrics table ─────────────────────────────────────
    print(f"\n{'='*116}")
    print("ABLATION TABLE — Leave-one-out (component flags + metrics)")
    print(f"{'='*116}")
    print(f"{'Variant':14} | {'MAD':>3} | {'Pol':>3} | {'Multi':>5} | "
          f"{'Spat':>4} | {'Fidelity':>8} | {'SNR':>6} | {'Prec':>5} | "
          f"{'Onset ms':>8} | {'Energy µJ':>9} | {'Sparsity':>8}")
    print("-" * 116)
    yn = lambda b: '✔' if b else '✘'
    for name, r in results.items():
        m, p, ms, sg = ABLATION_FLAGS[name]
        lat = f"{r['latency_ms']:.1f}" if r['latency_ms'] is not None else "N/A"
        marker = ' ←' if name == 'Full AMSTE' else '  '
        print(f"{name:14} | {yn(m):>3} | {yn(p):>3} | {yn(ms):>5} | "
              f"{yn(sg):>4} | {r['fidelity']:>8.3f} | {r['snr']:>6.2f} | "
              f"{r['precision']:>5.3f} | {lat:>8} | {r['energy_uj']:>9.3f} | "
              f"{r['sparsity']:>7.2%}{marker}")
    print(f"\n  ✔ = component active     ✘ = component removed (ablated)")
    print(f"  Full AMSTE row marked with ← (proposed full method)")

    # ── Diagnostic interpretation hints ──────────────────────────────────────
    full = results['Full AMSTE']
    print(f"\n     Diagnostic notes (single file — confirm with batch sweep, notebook 2):")
    print(f"     • Removing MAD            → expect noisier-channel false spikes, lower precision")
    print(f"     • Removing polarity       → expect lower recall on rarefaction half-cycles")
    print(f"     • Removing multi-scale    → expect higher FNR (single-lag noise sensitivity)")
    print(f"     • Removing spatial gate   → expect higher spike rate / energy")
    print(f"     • Full AMSTE              → best balance of detection vs efficiency")
    print(f"\n     Single-file metrics are diagnostic only.")
    print(f"     Paper-quality F1 / Recall / FNR / AUPRC come from batch SNN training (notebook 4).")

    return results


# =============================================================================
# 7. VISUALISATION
# =============================================================================
def _save(fig, name):
    path = os.path.join(OUTPUT_DIR, name)
    fig.savefig(path, dpi=300, bbox_inches='tight')
    print(f" Saved: {name}")
    plt.show()


def plot_segy_section_all_variants(filtered, norm_env, dt_ms, variants):
    """SEGY section + spike rasters for the 6 ablation variants."""
    n_tr, n_smp = filtered.shape
    t_axis      = np.arange(n_smp) * dt_ms
    p1, p99     = np.percentile(filtered, [1, 99])
    vmax        = max(abs(p1), abs(p99))
    n_v         = len(variants)

    height_ratios = [3] + [1] * n_v
    fig = plt.figure(figsize=(18, 4 + 2.0 * n_v))
    gs  = gridspec.GridSpec(n_v + 1, 1, height_ratios=height_ratios, hspace=0.06)

    ax0 = fig.add_subplot(gs[0])
    im  = ax0.imshow(filtered.T, aspect='auto', cmap='seismic',
                     vmin=-vmax, vmax=vmax,
                     extent=[0, n_tr, t_axis[-1], t_axis[0]])
    ax0.set_title('SEGY Section — bandpass filtered (50–250 Hz)',
                  fontsize=13, fontweight='bold')
    ax0.set_ylabel('Time (ms)')
    ax0.axhspan(EVENT_START_MS, EVENT_END_MS, color='yellow', alpha=0.20,
                label='Event window')
    ax0.legend(fontsize=9, loc='upper right')
    plt.colorbar(im, ax=ax0, label='Amplitude', fraction=0.015)

    for row, (name, spk) in enumerate(variants.items()):
        fid = temporal_fidelity(spk, norm_env, dt_ms)
        snr = event_snr(spk, dt_ms)
        sp  = 100.0 * np.sum(np.abs(spk)) / spk.size
        col = ENCODER_COLORS.get(name, 'gray')
        ax  = fig.add_subplot(gs[row + 1], sharex=ax0)
        tr_idx, t_idx = np.where(np.abs(spk) > 0)
        if len(t_idx):
            ax.scatter(tr_idx, t_idx * dt_ms, c=col, s=1.4, alpha=0.55,
                       linewidths=0, rasterized=True)
        else:
            ax.text(0.5, 0.5, 'No spikes', ha='center', va='center',
                    transform=ax.transAxes, color='gray', fontsize=9)
        ax.axhspan(EVENT_START_MS, EVENT_END_MS, color='yellow', alpha=0.18)
        ax.set_ylim(t_axis[-1], t_axis[0])
        ax.set_ylabel('ms', fontsize=7)
        ax.yaxis.set_tick_params(labelsize=7)
        ax.grid(True, alpha=0.22, lw=0.4)
        label = (f"{name}  |  Fidelity={fid:.3f}  SNR={snr:.1f}×  "
                 f"Sparsity={sp:.2f}%")
        ax.text(0.005, 0.96, label, transform=ax.transAxes, fontsize=8.5,
                fontweight='bold', va='top', color=col,
                bbox=dict(facecolor='white', alpha=0.72, edgecolor='none', pad=2))
        if row < n_v - 1:
            plt.setp(ax.get_xticklabels(), visible=False)
        else:
            ax.set_xlabel('Trace index', fontsize=11)

    fig.suptitle('SEGY Section + Spike Rasters — AMSTE Ablation (V1–V6)',
                 fontsize=13, y=1.002, fontweight='bold')
    _save(fig, 'amste_ablation_segy_section.png')
    return fig


def plot_single_trace_all_variants(filtered, dt_ms, variants,
                                    trace_idx=DISPLAY_TRACE):
    """Single-trace waveform + spike trains for the 6 ablation variants."""
    n_smp  = filtered.shape[1]
    t_axis = np.arange(n_smp) * dt_ms
    env_tr  = np.abs(hilbert(filtered[trace_idx]))
    env_tr /= (np.max(env_tr) + 1e-9)
    amp_max = np.max(np.abs(filtered[trace_idx]))
    n_v     = len(variants)
    ev_s    = max(0, int(EVENT_START_MS / dt_ms))
    ev_e    = min(n_smp, int(EVENT_END_MS / dt_ms))

    height_ratios = [2.5] + [0.9] * n_v
    fig = plt.figure(figsize=(16, 3.5 + 1.4 * n_v))
    gs  = gridspec.GridSpec(n_v + 1, 1, height_ratios=height_ratios, hspace=0.08)

    ax_wave = fig.add_subplot(gs[0])
    ax_wave.plot(t_axis, filtered[trace_idx], color='steelblue', lw=0.75,
                 label=f'Trace {trace_idx}')
    ax_wave.fill_between(t_axis, env_tr * amp_max, -env_tr * amp_max,
                         alpha=0.18, color='darkorange', label='Envelope ±')
    ax_wave.axvspan(EVENT_START_MS, EVENT_END_MS, color='yellow', alpha=0.28,
                    label='Event window')
    ax_wave.set_title(f'Trace {trace_idx} — Filtered Waveform (50–250 Hz)',
                      fontsize=12, fontweight='bold')
    ax_wave.set_ylabel('Amplitude')
    ax_wave.legend(fontsize=9, loc='upper right', ncol=3)
    ax_wave.grid(True, alpha=0.28)

    for row, (name, spk) in enumerate(variants.items()):
        col     = ENCODER_COLORS.get(name, 'gray')
        spk_tr  = np.where(np.abs(spk[trace_idx]) > 0)[0]
        times   = spk_tr * dt_ms
        n_total = len(times)
        n_event = int(np.sum(np.abs(spk[trace_idx, ev_s:ev_e]) > 0))
        e_uj    = n_total * ENERGY_PER_SPIKE_PJ / 1e6

        ax = fig.add_subplot(gs[row + 1], sharex=ax_wave)
        if n_total > 0:
            ax.eventplot(times, lineoffsets=0.5, linelengths=0.80,
                         colors=col, linewidths=0.9)
        else:
            ax.text(0.5, 0.5, '— 0 spikes —', ha='center', va='center',
                    transform=ax.transAxes, color='gray', fontsize=9, style='italic')
        ax.axvspan(EVENT_START_MS, EVENT_END_MS, color='yellow', alpha=0.28)
        ax.set_yticks([])
        ax.set_ylim(0, 1)
        ax.grid(True, alpha=0.22, axis='x', lw=0.45)
        ax.set_ylabel(name, fontsize=8, rotation=0, labelpad=80, va='center',
                      color=col, fontweight='bold')
        ann = f"total={n_total}  in-event={n_event}  E={e_uj:.3f}µJ"
        ax.text(0.998, 0.90, ann, transform=ax.transAxes, fontsize=8,
                va='top', ha='right', color=col,
                bbox=dict(facecolor='white', alpha=0.72, edgecolor='none', pad=1.5))
        if row < n_v - 1:
            plt.setp(ax.get_xticklabels(), visible=False)
        else:
            ax.set_xlabel('Time (ms)', fontsize=11)

    fig.suptitle(f'Trace {trace_idx} — Spike Trains: AMSTE Ablation (V1–V6)',
                 fontsize=13, y=1.002, fontweight='bold')
    _save(fig, f'amste_ablation_trace_{trace_idx}.png')
    return fig


def plot_temporal_fidelity_comparison(norm_envelope, dt_ms, variants):
    """Stacked envelope reference + spike-density profile per variant."""
    t_axis   = np.arange(norm_envelope.shape[1]) * dt_ms
    env_ref  = np.mean(norm_envelope, axis=0)
    k        = max(3, int(20 / dt_ms))
    env_sm   = np.convolve(env_ref, np.ones(k) / k, mode='same')
    n_v      = len(variants)

    fig, axs = plt.subplots(n_v + 1, 1, figsize=(14, 2.4 * (n_v + 1)), sharex=True)
    axs[0].plot(t_axis, env_sm, color='black', lw=1.5)
    axs[0].axvspan(EVENT_START_MS, EVENT_END_MS, alpha=0.2, color='yellow',
                   label='Event window')
    axs[0].set_title('Stacked Envelope (Reference)',
                     fontsize=11, fontweight='bold')
    axs[0].set_ylabel('Norm. Amplitude')
    axs[0].legend(fontsize=9)
    axs[0].grid(True, alpha=0.3)

    for i, (name, spk) in enumerate(variants.items()):
        profile = spike_density_profile(spk, dt_ms)
        fid     = temporal_fidelity(spk, norm_envelope, dt_ms)
        snr     = event_snr(spk, dt_ms)
        col     = ENCODER_COLORS.get(name, 'gray')
        axs[i + 1].fill_between(t_axis, profile, alpha=0.6, color=col)
        axs[i + 1].axvspan(EVENT_START_MS, EVENT_END_MS, alpha=0.15, color='yellow')
        axs[i + 1].set_title(
            f'{name}  |  Fidelity={fid:.3f}  SNR={snr:.2f}×',
            fontsize=11, fontweight='bold')
        axs[i + 1].set_ylabel('Spike Density')
        axs[i + 1].grid(True, alpha=0.3)

    axs[-1].set_xlabel('Time (ms)', fontsize=11)
    plt.suptitle('Temporal Fidelity: Spike Density vs Event Envelope — '
                 'AMSTE Ablation (V1–V6)',
                 fontsize=13, y=1.01)
    plt.tight_layout()
    _save(fig, 'amste_ablation_temporal_fidelity.png')


def plot_spike_statistics(results):
    """Bar charts: Fidelity / Precision / SNR / Efficiency / Energy / Sparsity."""
    names      = list(results.keys())
    fidelity   = [results[n]['fidelity']     for n in names]
    snr        = [results[n]['snr']          for n in names]
    precision  = [results[n]['precision']    for n in names]
    energy     = [results[n]['energy_uj']    for n in names]
    sparsity   = [results[n]['sparsity']*100 for n in names]
    efficiency = [results[n]['efficiency']   for n in names]
    x          = np.arange(len(names))
    w          = 0.55
    colors     = [ENCODER_COLORS.get(n, 'gray') for n in names]
    event_fraction = (EVENT_END_MS - EVENT_START_MS) / (WINDOW_END_MS - WINDOW_START_MS)

    fig, axs = plt.subplots(3, 2, figsize=(16, 13))
    for ax, vals, title, hline, hlabel in [
        (axs[0,0], fidelity,   'Temporal Fidelity (↑)',     0.7,  'Target 0.70'),
        (axs[0,1], precision,  'Event Precision (↑)',        event_fraction,
                                                              f'Chance ({event_fraction:.2f})'),
        (axs[1,0], snr,        'Event SNR (↑)',              2.0,  'Target 2×'),
        (axs[1,1], efficiency, 'Efficiency fidelity/µJ (↑)', None, None),
        (axs[2,0], energy,     'Energy Cost µJ (↓)',         None, None),
        (axs[2,1], sparsity,   'Sparsity % (↓)',             None, None),
    ]:
        bars = ax.bar(x, vals, width=w, color=colors)
        ax.set_title(title, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(names, rotation=35, ha='right')
        if hline is not None:
            ax.axhline(hline, ls='--', color='red', label=hlabel)
            ax.legend()
        # Highlight Full AMSTE (proposed full method) and Basic (lower bound)
        for idx, name in enumerate(names):
            if name == 'Full AMSTE':
                bars[idx].set_edgecolor('black')
                bars[idx].set_linewidth(2.5)
            elif name == 'Basic':
                bars[idx].set_edgecolor('dimgray')
                bars[idx].set_linewidth(1.5)
                bars[idx].set_linestyle('--')

    plt.suptitle('AMSTE Ablation — Per-Variant Metrics (V1–V6)\n'
                 'Black border = Full AMSTE (proposed)  |  Dashed border = Basic (lower bound)',
                 fontsize=12, y=1.01)
    plt.tight_layout()
    _save(fig, 'amste_ablation_metrics.png')


# =============================================================================
# 8. MAIN
# =============================================================================
if __name__ == "__main__":
    print("=" * 78)
    print("DAS MICROSEISMIC SNN ENCODING — AMSTE ABLATION (single-file diagnostic)")
    print("Variants: V1 Basic | V2 No-MAD | V3 No-Polarity | V4 No-MultiScale")
    print("          V5 No-Spatial | V6 Full AMSTE (proposed)")
    print("=" * 78)

    if not os.path.exists(FILE_PATH):
        raise FileNotFoundError(f"File not found: {FILE_PATH}")

    data, dt_ms = load_segy_file(FILE_PATH)
    if data is None:
        raise RuntimeError("Failed to load SEGY file")

    print(f"\n Bandpass {FILTER_LOW_HZ}–{FILTER_HIGH_HZ} Hz…")
    filtered = bandpass_filter(data, dt_ms)
    envelope = extract_envelope(filtered)

    if EXTRACT_WINDOW:
        print(f"  Window {WINDOW_START_MS}–{WINDOW_END_MS} ms…")
        filtered, _, _ = extract_time_window(filtered, dt_ms,
                                              WINDOW_START_MS, WINDOW_END_MS)
        envelope, _, _ = extract_time_window(envelope, dt_ms,
                                              WINDOW_START_MS, WINDOW_END_MS)

    norm_env    = normalize_unit(envelope)        # [0, 1] envelope reference
    norm_signed = normalize_signed(filtered)      # [-1, +1] AMSTE input

    stacked    = np.mean(norm_env, axis=0)
    event_time = np.argmax(stacked) * dt_ms
    print(f" Event centre: {event_time:.1f} ms "
          f"(window: {EVENT_START_MS}–{EVENT_END_MS} ms)")
    print(f"   Note: single-file diagnostic window only.")
    print(f"   Batch experiments use per-file labels (not this hard-coded window).")

    print("\n Building AMSTE ablation variants…")
    all_variants = _build_amste_ablation_variants(norm_signed, dt_ms)

    print("\n Plot 1 — SEGY section + ablation rasters…")
    plot_segy_section_all_variants(filtered, norm_env, dt_ms, all_variants)

    print(f"\n Plot 2 — Single-trace spike trains (trace {DISPLAY_TRACE})…")
    plot_single_trace_all_variants(filtered, dt_ms, all_variants,
                                    trace_idx=DISPLAY_TRACE)

    print("\n Plot 3 — Temporal fidelity comparison…")
    plot_temporal_fidelity_comparison(norm_env, dt_ms, all_variants)

    print("\n Plot 4 — Ablation experiment + leave-one-out table…")
    experiment_results = run_ablation_experiment(all_variants, norm_env, dt_ms)
    print("""
   PAPER-WORDING TEMPLATE:
  ────────────────────────────────────────────────────────────────────────
  "To examine the contribution of each AMSTE component, an ablation study
   was performed by removing one component at a time while keeping the SNN
   architecture, training procedure, data split, and hyperparameters
   unchanged. The evaluated variants include AMSTE without adaptive MAD
   thresholding, without polarity separation, without multi-scale temporal
   voting, and without the spatial coherence gate. This isolates the
   effect of each design choice on event sensitivity, false-negative rate,
   spike sparsity, and energy consumption."
  ────────────────────────────────────────────────────────────────────────
""")

    print("\n Plot 5 — Bar-chart summary of ablation metrics…")
    plot_spike_statistics(experiment_results)

    print("\n Saving results…")
    np.save(os.path.join(OUTPUT_DIR, 'amste_ablation_variants.npy'), all_variants)
    np.save(os.path.join(OUTPUT_DIR, 'filtered_data.npy'),           filtered)
    np.save(os.path.join(OUTPUT_DIR, 'amste_ablation_results.npy'),  experiment_results)
    print(" Done.")
